# 기온 예측 · PyTorch GPU 학습
강남 AWS 지점 400의 **1시간 후, 24시간 후, 72시간 후 기온(℃)**을 각각 예측하는 모델입니다.
강수 확률 노트북과 함께 사용합니다. 커널은 프로젝트의 `.venv`를 선택하고 위부터 실행하세요.
학습은 2015~2022년, 검증은 2023~2024년입니다. 2025년 최종 평가 자료는 읽지 않습니다.
현재 모델은 세 예측 시점만 지원합니다. 매시간 이어지는 72시간 예보는 추가 모델과 평가가 필요합니다.
원래 서비스의 기온 예측을 위한 첫 후보이며, 검증 결과를 보고 개선해야 합니다.

## 1. 라이브러리와 GPU 설정

In [ ]:
import sys
import json
import runpy
from pathlib import Path
from datetime import datetime
from time import perf_counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import sklearn
import joblib

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
print("사용 장치:", device, " / PyTorch:", torch.__version__)
print("파이썬:", sys.executable)


## 2. 기온 정답 연결
정답은 다음 24시간 평균이 아니라 각 예측 시점에 측정된 기온입니다.
같은 시간의 입력과 정답을 연결하고, 미래 정답 칼럼은 입력에서 제외합니다.
현재·미래 기온 정답이 모두 관측된 행을 사용합니다. 다른 입력의 결측은 다음 셀에서 처리합니다.

In [ ]:
STATION = 400
HORIZONS = [1, 24, 72]
candidates = [
    Path.cwd() / "weather_data_v2", Path.cwd(),
    Path(r"C:\Users\jeonm\Documents\Codex\2026-10-12\wlrm\outputs\weather_data_v2"),
]
DATA_DIR = None
for candidate in candidates:
    path = candidate / "model_input_columns.json"
    if path.is_file():
        candidate_config = json.loads(path.read_text(encoding="utf-8"))
        if candidate_config.get("version") == "aws-split-v2":
            DATA_DIR, config = candidate, candidate_config
            break
if DATA_DIR is None:
    raise FileNotFoundError("weather_data_v2 폴더를 프로젝트 폴더에 넣어주세요.")

helper = runpy.run_path(str(DATA_DIR / "check_data.py"))
feature_candidates = config["model_input_columns"]
assert not set(feature_candidates) & set(config["forbidden_input_columns"])
target_columns = ["station_id", "observed_at"]
for horizon in HORIZONS:
    target_spec = config["temperature"][str(horizon)]
    target_columns.extend(target_spec.values())

joined_splits, observed_parts = {}, []
for split in ("train", "validation"):
    spec = config["splits"][split]
    observations = helper["read_station"](
        DATA_DIR / spec["input_file"], STATION,
        list(dict.fromkeys(["station_id", "station_name", "observed_at", *feature_candidates]))
    )
    targets = helper["read_station"](
        DATA_DIR / spec["target_file"], STATION, list(dict.fromkeys(target_columns))
    )
    joined = observations.merge(targets, on=["station_id", "observed_at"],
                                how="left", validate="one_to_one", indicator=True)
    assert joined["_merge"].eq("both").all()
    assert observations.observed_at.diff().dropna().eq(pd.Timedelta(hours=1)).all()
    joined_splits[split] = joined
    observed_parts.append(observations[["observed_at", "temperature_c"]])
    print(split, f"입력 {len(observations):,}행")
observed_temperature = pd.concat(observed_parts).set_index("observed_at")["temperature_c"]
assert observed_temperature.index.is_unique
print("데이터 폴더:", DATA_DIR)


## 3. 예측 시간별 데이터로더
각 모델은 현재 기온에서 미래 기온까지의 변화량을 학습하고, 예측할 때 현재 기온을 더합니다.
입력 결측 중앙값·정규화와 정답 변화량 정규화는 각 모델의 학습 자료에서만 계산합니다.
학습에서 전부 결측인 칼럼만 제외하고 나머지 칼럼과 결측 표시를 보존합니다.

In [ ]:
prepared = {}
for horizon in HORIZONS:
    spec = config["temperature"][str(horizon)]
    selected = {}
    for split, joined in joined_splits.items():
        rows = joined.loc[joined[spec["eligible"]].eq(1)].reset_index(drop=True)
        assert not rows.empty
        assert np.isfinite(rows[["temperature_c", spec["target"]]].to_numpy()).all()
        assert rows[spec["target_time"]].eq(rows.observed_at + pd.Timedelta(hours=horizon)).all()
        assert rows[spec["target_time"]].le(pd.Timestamp(config["splits"][split]["end_at"])).all()
        selected[split] = rows
    train, validation = selected["train"], selected["validation"]
    features = [c for c in feature_candidates if train[c].notna().any()]
    dropped = [c for c in feature_candidates if c not in features]
    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    target_scaler = StandardScaler()
    x_train = preprocessor.fit_transform(train[features]).astype(np.float32)
    x_val = preprocessor.transform(validation[features]).astype(np.float32)
    train_delta = (train[spec["target"]] - train.temperature_c).to_numpy().reshape(-1, 1)
    val_delta = (validation[spec["target"]] - validation.temperature_c).to_numpy().reshape(-1, 1)
    z_train = target_scaler.fit_transform(train_delta).astype(np.float32)
    z_val = target_scaler.transform(val_delta).astype(np.float32)
    assert all(np.isfinite(a).all() for a in (x_train, x_val, z_train, z_val))
    train_dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(z_train))
    val_dataset = TensorDataset(torch.from_numpy(x_val), torch.from_numpy(z_val))
    prepared[horizon] = {
        "train":train, "validation":validation, "features":features, "dropped":dropped,
        "preprocessor":preprocessor, "target_scaler":target_scaler, "input_size":x_train.shape[1],
        "train_loader":DataLoader(train_dataset, batch_size=256, shuffle=True,
            num_workers=0, pin_memory=(device.type == "cuda"),
            generator=torch.Generator().manual_seed(SEED + horizon)),
        "val_loader":DataLoader(val_dataset, batch_size=1024, shuffle=False,
            num_workers=0, pin_memory=(device.type == "cuda")),
    }
    print(f"{horizon}시간 후: 학습 {len(train):,}행 / 검증 {len(validation):,}행 / 입력 {len(features)}개")


## 4. 기온 회귀 모델
출력은 하나의 수치이며 sigmoid를 사용하지 않습니다. 손실은 기온 변화량의 제곱 오차입니다.
[PyTorch MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)

In [ ]:
HIDDEN_SIZES, DROPOUT = [128, 64], 0.1
def make_model(input_size):
    return nn.Sequential(
        nn.Linear(input_size, HIDDEN_SIZES[0]), nn.ReLU(), nn.Dropout(DROPOUT),
        nn.Linear(HIDDEN_SIZES[0], HIDDEN_SIZES[1]), nn.ReLU(),
        nn.Linear(HIDDEN_SIZES[1], 1),
    )

def run_epoch(model, loader, optimizer=None):
    model.train(optimizer is not None)
    total_loss, predictions = 0.0, []
    with torch.set_grad_enabled(optimizer is not None):
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
            output = model(inputs)
            loss = nn.functional.mse_loss(output, labels)
            if not torch.isfinite(loss):
                raise RuntimeError("학습 손실이 유효한 숫자가 아닙니다.")
            if optimizer is not None:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            if optimizer is None:
                predictions.append(output.detach().cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(predictions) if predictions else None
print(make_model(prepared[HORIZONS[0]]["input_size"]))


## 5. 세 기온 모델 학습
각각 최대 20회 학습합니다. 검증 오차가 5회 연속 개선되지 않으면 멈춥니다.
검증 오차가 가장 작았던 가중치를 복원합니다. 출력 RMSE의 단위는 ℃입니다.

In [ ]:
EPOCHS = 20
PATIENCE = 5
results = {}
for horizon in HORIZONS:
    torch.manual_seed(SEED + horizon)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED + horizon)
    data = prepared[horizon]
    model = make_model(data["input_size"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_loss, best_state, best_epoch, stale = float("inf"), None, 0, 0
    history, started = [], perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss, _ = run_epoch(model, data["train_loader"], optimizer)
        val_loss, _ = run_epoch(model, data["val_loader"])
        val_rmse = float(np.sqrt(val_loss) * data["target_scaler"].scale_[0])
        history.append({"epoch":epoch, "train_loss":train_loss, "val_loss":val_loss, "val_rmse_c":val_rmse})
        if val_loss < best_loss:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale += 1
        print(f"{horizon:2d}시간 | Epoch {epoch:02d} | 검증 RMSE {val_rmse:.3f}℃", flush=True)
        if stale >= PATIENCE:
            print("검증 오차 개선이 멈춰 조기 종료합니다.")
            break
    model.load_state_dict(best_state)
    model.eval()
    _, normalized_prediction = run_epoch(model, data["val_loader"])
    predicted_delta = data["target_scaler"].inverse_transform(normalized_prediction).ravel()
    prediction = data["validation"].temperature_c.to_numpy() + predicted_delta
    results[horizon] = {"model":model, "weights":best_state, "best_epoch":best_epoch,
                        "history":history, "prediction":prediction}
    print(f"{horizon}시간 모델 완료: 선택 Epoch {best_epoch}, {perf_counter()-started:.1f}초")


## 6. 기온 오차와 기준 예측 비교
기온은 정확도 % 대신 MAE·RMSE·편향(평균 예측−실제)을 ℃ 단위로 확인합니다.
현재 기온 유지와 최근 관측된 같은 시각 기온 반복을 비교합니다.
같은 시각 기준은 1시간 예측에서 t−23시간, 24·72시간 예측에서 t의 기온입니다.
미래의 관측값은 기준 예측에도 사용하지 않습니다. 24·72시간에서는 두 기준 예측이 같습니다.
비교 표는 각 예측 시간에서 모든 기준 값이 관측된 공통 행으로 계산합니다.

In [ ]:
metric_rows, examples_by_horizon = [], {}
for horizon in HORIZONS:
    data, result = prepared[horizon], results[horizon]
    rows = data["validation"]
    spec = config["temperature"][str(horizon)]
    actual = rows[spec["target"]].to_numpy()
    persistence = rows.temperature_c.to_numpy()
    offset = horizon - int(np.ceil(horizon / 24)) * 24
    baseline_times = rows.observed_at + pd.Timedelta(hours=offset)
    assert baseline_times.le(rows.observed_at).all()
    daily_repeat = observed_temperature.reindex(pd.DatetimeIndex(baseline_times)).to_numpy()
    common = np.isfinite(daily_repeat) & np.isfinite(actual) & np.isfinite(result["prediction"])
    assert common.any()
    for name, prediction in [("신경망", result["prediction"]), ("현재 기온 유지", persistence), ("같은 시각 기온 반복", daily_repeat)]:
        metric_rows.append({
            "예측 시간(h)":horizon, "모델":name, "평가 행 수":int(common.sum()),
            "MAE(℃)":float(mean_absolute_error(actual[common], prediction[common])),
            "RMSE(℃)":float(np.sqrt(mean_squared_error(actual[common], prediction[common]))),
            "편향(℃)":float(np.mean(prediction[common] - actual[common])),
        })
    result["all_valid_mae_c"] = float(mean_absolute_error(actual, result["prediction"]))
    result["all_valid_rmse_c"] = float(np.sqrt(mean_squared_error(actual, result["prediction"])))
    examples_by_horizon[horizon] = pd.DataFrame({
        "검증 기준 시각":rows.observed_at,
        "예측 대상 시각":rows[spec["target_time"]],
        "현재 기온(℃)":persistence, "예측 기온(℃)":result["prediction"], "실제 기온(℃)":actual,
    })
metrics_table = pd.DataFrame(metric_rows)
print(metrics_table.round(3).to_string(index=False))
print("24시간 후 기온 예측 예시 (과거 검증 시점)")
print(examples_by_horizon[24].tail(5).round({
    "현재 기온(℃)":2, "예측 기온(℃)":2, "실제 기온(℃)":2,
}).to_string(index=False))


## 7. 모델과 전처리 저장
실행 폴더의 models 아래 새 폴더에 시간별 모델을 저장합니다. 가중치는 CPU 텐서로 저장합니다.
API에서는 저장된 입력 순서·전처리·변화량 정규화를 그대로 사용해야 합니다.
예측 기온 = target_scaler.inverse_transform(모델 출력) + 현재 관측 기온입니다.
검증에서 기준보다 성능이 나쁘면 개선한 뒤 사용하세요. 2025년 최종 평가와 Docker API 연결은 별도 단계입니다.

In [ ]:
RUN_DIR = Path.cwd() / "models" / f"temperature_pytorch_{STATION}_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
metrics_table.to_csv(RUN_DIR / "validation_metrics.csv", index=False, encoding="utf-8-sig")
for horizon in HORIZONS:
    data, result = prepared[horizon], results[horizon]
    folder = RUN_DIR / f"plus_{horizon}h"
    folder.mkdir()
    torch.save(result["weights"], folder / "model_weights.pt")
    joblib.dump(data["preprocessor"], folder / "preprocessor.joblib")
    joblib.dump(data["target_scaler"], folder / "target_scaler.joblib")
    pd.DataFrame(result["history"]).to_csv(folder / "training_history.csv", index=False, encoding="utf-8-sig")
    examples_by_horizon[horizon].to_csv(folder / "validation_predictions.csv", index=False, encoding="utf-8-sig")
    model_config = {
        "station_id":STATION, "task":"temperature", "horizon_hours":horizon,
        "model_type":"MLP", "input_size":data["input_size"],
        "hidden_sizes":HIDDEN_SIZES, "dropout":DROPOUT,
        "feature_columns":data["features"], "dropped_columns":data["dropped"],
        "target":config["temperature"][str(horizon)]["target"],
        "output_mode":"normalized_temperature_change_from_origin",
        "inverse_prediction":"target_scaler.inverse_transform(network_output) + temperature_c_at_origin",
        "unit":"celsius", "best_epoch":result["best_epoch"], "seed":SEED + horizon,
        "validation_mae_c":result["all_valid_mae_c"], "validation_rmse_c":result["all_valid_rmse_c"],
        "dataset_version":config["version"], "train_period":config["splits"]["train"],
        "validation_period":config["splits"]["validation"], "test_evaluated":False,
        "training_device":str(device), "torch_version":torch.__version__,
        "sklearn_version":sklearn.__version__, "numpy_version":np.__version__, "pandas_version":pd.__version__,
    }
    (folder / "model_config.json").write_text(json.dumps(model_config, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장한 폴더:", RUN_DIR.resolve())
